[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 03](README.md)

# OpenMP: fork-join y entorno de datos

**Tema:** 03 · **Sesiones:** 11, 12 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Qué variables comparte cada hilo y cuáles deben ser privadas para conservar corrección?


## Resultados de aprendizaje

- Explicar fork-join y equipos de hilos.
- Auditar `shared`, `private`, `firstprivate` y reducciones.
- Usar `default(none)` como herramienta de revisión.


## Modelo conceptual

OpenMP crea equipos alrededor de regiones paralelas y sincroniza implícitamente salvo cláusula contraria.

El alcance léxico de C/C++ no basta para deducir el atributo de datos de OpenMP.

Afinidad y schedule afectan localidad, pero no deben cambiar el resultado correcto.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "03"
NOTEBOOK = "03_openmp/01_modelo_datos.ipynb"
assert (ROOT / "curso" / "notebooks" / "03_openmp" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Auditoría de variables

Se documenta el rol de cada dato en una suma paralela.


In [ ]:
variables = {
    "entrada": ("shared", "solo lectura"),
    "n": ("shared", "límite inmutable"),
    "i": ("private", "índice de iteración"),
    "parcial": ("private", "acumulador por hilo"),
    "total": ("reduction", "combinación asociativa definida"),
}
assert variables["i"][0] == "private"
for name, (scope, reason) in variables.items(): print(f"{name:8} {scope:10} {reason}")


**Interpretación.** La auditoría convierte `default(none)` en una explicación del diseño y no solo en una exigencia sintáctica.


## Schedule estático

Se visualiza la asignación de iteraciones por bloques contiguos.


In [ ]:
n, threads = 19, 4
owner = {}
q, r = divmod(n, threads)
start = 0
for thread in range(threads):
    end = start + q + (thread < r)
    for i in range(start, end): owner[i] = thread
    start = end
assert sorted(owner) == list(range(n))
for thread in range(threads): print(thread, [i for i in owner if owner[i] == thread])


**Interpretación.** La implementación de `schedule(static)` puede distribuir chunks según la cláusula; se documenta la forma usada en el experimento.


## Práctica reproducible

1. Compilar `openmp/hello.cc` y observar identificadores de hilo.
2. Agregar `default(none)` a un bucle y clasificar todas las variables.
3. Registrar `OMP_NUM_THREADS`, afinidad y schedule.


## Errores frecuentes

- Asumir que toda variable local es private.
- Escribir salida concurrente y usar su orden como evidencia.
- Cambiar schedule y tamaño a la vez.

## Criterios de aceptación

- Clasificación explícita de datos.
- Salida igual a referencia serial.
- Entorno OpenMP conservado en el informe.


## Referencias y material relacionado

- [Hello OpenMP](../../../openmp/hello.cc)
- [Data sharing](../../../openmp/data_sharing.c)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 03](README.md)
